In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Options: 0 = all, 1 = filter INFO, 2 = filter WARNING, 3 = ERROR only
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Set memory growth to True
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [2]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [3]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, Callback
from tensorflow.keras.optimizers import Adam
import functools


from IPython.display import display

In [4]:
def timeit(func):
    """Decorator to measure execution time of a function and print a friendly summary."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        mins, secs = divmod(elapsed, 60)
        hours, mins = divmod(mins, 60)
        time_str = (
            f"{int(hours)}h {int(mins)}m {secs:.2f}s"
            if hours
            else f"{int(mins)}m {secs:.2f}s"
            if mins
            else f"{secs:.2f}s"
        )
        print(f"\n🚀 Script completed successfully in {time_str}!")
        return result

    return wrapper


def prep_pc():
    # Quiet TensorFlow logging
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    import tensorflow as tf

    # GPU memory growth
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)


class TimeStopping(Callback):
    """Stop training when exceeding a time limit (in seconds)."""

    def __init__(self, max_seconds=300):
        super().__init__()
        self.max_seconds = max_seconds
        self.start_time = None

    def on_train_begin(self, logs=None):
        self.start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        if time.time() - self.start_time > self.max_seconds:
            print(
                f"\n⏱️ Stopping training - time limit of {self.max_seconds} seconds reached."
            )
            self.model.stop_training = True


def eval_classification(model, X, y, name="model", labels=None, plot=False):
    """Evaluate and optionally plot a confusion matrix."""
    preds = np.rint(model.predict(X))
    print(f"\n=== {name} ===")
    print(classification_report(y, preds, target_names=labels))

    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    score_df = pd.DataFrame(
        index=[name],
        data={
            "Accuracy": [accuracy_score(y, preds)],
            "Precision": [precision_score(y, preds)],
            "Recall": [recall_score(y, preds)],
            "F1 Score": [f1_score(y, preds)],
            "FPR": [fp / (fp + tn) if (fp + tn) > 0 else np.nan],
            "FNR": [fn / (fn + tp) if (fn + tp) > 0 else np.nan],
        },
    )

    if plot:
        ConfusionMatrixDisplay.from_predictions(y, preds, display_labels=labels)
        plt.title(name)
        plt.show()

    return score_df


def plot_history(history, plot=False):
    """Plot training & validation metrics from a Keras History.
    Epoch axis is 1-based and corresponds to the actual epochs run (may be < max_epochs if early-stopping).
    """
    if not plot:
        return

    # Determine number of epochs actually run
    first_metric = next(m for m in history.history.keys() if not m.startswith("val_"))
    n_epochs = len(history.history[first_metric])
    epochs = range(1, n_epochs + 1)

    for metric in history.history:
        if metric.startswith("val_"):
            continue
        plt.plot(epochs, history.history[metric], label=metric)
        val_metric = f"val_{metric}"
        if val_metric in history.history:
            plt.plot(epochs, history.history[val_metric], label=val_metric)
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.title(metric)
        plt.xticks(list(epochs))
        plt.legend()
        plt.grid(True)
        plt.show()


### Classification:
To evaluate the model's performance, the F1 Score is used. It is suitable for phishing detection because it strikes a balance between precision and recall, considering both false positives and false negatives, rather than focusing on one aspect alone. Unlike accuracy, it treats imbalanced datasets well and prevents incorrect performance conclusions. By encapsulating both types of error within a single metric, F1 provides a better understanding of the model's performance and confidence.

### Pseudo Code

FOR each layer_config in layer_configs:
  FOR each epoch_count in epochs_list:
    FOR each min_delta in min_deltas:
      FOR each patience_value in patiences:
        Build sequential model
        Add input + hidden layers with relu
        Add sigmoid output layer
        Compile with Adam, binary_crossentropy
        Apply EarlyStopping + TimeStopping
        Train on X_train, validate on X_val
        Evaluate on both sets using F1 Score, Accuracy, Precision, Recall
        If best F1, save model and config


In [5]:
# --- Load Data ---
df_train = pd.read_csv("Split_Data/Model_Ready/train.csv")
X_train = df_train.drop(columns="label")
y_train = df_train["label"]
df_val = pd.read_csv("Split_Data/Model_Ready/val.csv")
X_val = df_val.drop(columns="label")
y_val = df_val["label"]
df_test = pd.read_csv("Split_Data/Model_Ready/test.csv")
X_test = df_test.drop(columns="label")
y_test = df_test["label"]

# --- Hyperparameter Space ---
layer_configs = [[64], [64, 64], [64, 128, 64], [256, 128, 128, 256]]
epochs_list = [50, 100, 200, 500]
min_deltas = [0.0, 0.001]
patiences = [3, 5, 7, 9]
dropout_rates = [0.3, 0.5]

best_f1 = -1
best_model = None
best_history = None
best_config = None
all_scores = []
labels = ["No Phishing Email", "Phishing Email"]



### Build de Model

In [ ]:
# --- Grid Search ---
for layers in layer_configs:
    for epochs in epochs_list:
        for min_delta in min_deltas:
            for patience in patiences:
                for dropout_rate in dropout_rates:
                    config_desc = (
                        f"layers={layers}, epochs={epochs}, min_delta={min_delta}, "
                        f"patience={patience}, dropout_rate={dropout_rate}"
                    )
                    print(f"\nConfig: {config_desc}")

                    # Build model
                    model = Sequential(
                        [
                            Dense(
                                layers[0],
                                activation="relu",
                                input_shape=(X_train.shape[1],),
                            ),
                            Dropout(dropout_rate),
                            *[
                                layer
                                for units in layers[1:]
                                for layer in (
                                    Dense(units, activation="relu"),
                                    Dropout(dropout_rate),
                                )
                            ],
                            Dense(1, activation="sigmoid"),
                        ]
                    )

                    model.compile(
                        optimizer=Adam(learning_rate=1e-3),
                        loss="binary_crossentropy",
                        metrics=[
                            tf.keras.metrics.BinaryAccuracy(name="acc"),
                            tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall"),
                            tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
                            tf.keras.metrics.AUC(curve="PR",  name="pr_auc"),
                        ],
                    )


                    early_stopping = EarlyStopping(
                        monitor="val_loss",
                        min_delta=min_delta,
                        patience=patience,
                        restore_best_weights=True,
                        verbose=1,
                        mode="min",
                    )
                    time_stopping = TimeStopping(max_seconds=300)

                    history = model.fit(
                        X_train,
                        y_train,
                        validation_data=(X_val, y_val),
                        epochs=epochs,
                        batch_size=32,
                        callbacks=[early_stopping, time_stopping],
                        verbose=2,
                    )

                    # Collect metrics without plotting
                    plot_history(history, plot=False)
                    train_scores = eval_classification(
                        model,
                        X_train,
                        y_train,
                        name=f"train {config_desc}",
                        labels=labels,
                        plot=False,
                    )
                    val_scores = eval_classification(
                        model,
                        X_val,
                        y_val,
                        name=f"val {config_desc}",
                        labels=labels,
                        plot=False,
                    )

                    all_scores.append(pd.concat([train_scores, val_scores]))

                    val_f1 = val_scores.iloc[0]["F1 Score"]
                    if val_f1 > best_f1:
                        best_f1 = val_f1
                        best_model = model
                        best_history = history
                        best_config = {
                            "layers": layers,
                            "epochs": epochs,
                            "min_delta": min_delta,
                            "patience": patience,
                            "dropout_rate": dropout_rate,
                            "val_f1_score": val_f1,
                        }

# --- Report & Save ---
print("\n🏆 Best Model Configuration:")
for k, v in best_config.items():
    print(f"{k}: {v}")

best_model.save("best_mlp_model_iter2.h5")
print("\n📁 Best model saved as 'best_mlp_model_iter2.h5'")

all_results_df = pd.concat(all_scores)
all_results_df.to_csv("mlp_model_results.csv", index=True)
print("Results saved to mlp_model_results.csv")

# --- Final Plots for Best Model ---
print("\n📊 Training History of Best Model:")
plot_history(best_history, plot=True)

print("\n🧪 Confusion Matrix & Metrics on Best Validation Set:")
eval_classification(
    best_model,
    X_val,
    y_val,
    name="Best Model on Validation",
    labels=labels,
    plot=True,
)



Config: layers=[64], epochs=50, min_delta=0.0, patience=3, dropout_rate=0.3
Epoch 1/50


/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1753284768.035982 3175435 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1753284768.036001 3175435 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


1650/1650 - 17s - 11ms/step - acc: 0.9299 - loss: 0.2034 - pr_auc: 0.9760 - precision: 0.9332 - recall: 0.9320 - roc_auc: 0.9774 - val_acc: 0.9597 - val_loss: 0.1178 - val_pr_auc: 0.9890 - val_precision: 0.9534 - val_recall: 0.9703 - val_roc_auc: 0.9902
Epoch 2/50
1650/1650 - 16s - 10ms/step - acc: 0.9487 - loss: 0.1436 - pr_auc: 0.9858 - precision: 0.9484 - recall: 0.9534 - roc_auc: 0.9869 - val_acc: 0.9603 - val_loss: 0.1143 - val_pr_auc: 0.9900 - val_precision: 0.9588 - val_recall: 0.9655 - val_roc_auc: 0.9908
Epoch 3/50
1650/1650 - 16s - 10ms/step - acc: 0.9508 - loss: 0.1390 - pr_auc: 0.9862 - precision: 0.9499 - recall: 0.9560 - roc_auc: 0.9874 - val_acc: 0.9598 - val_loss: 0.1139 - val_pr_auc: 0.9904 - val_precision: 0.9576 - val_recall: 0.9658 - val_roc_auc: 0.9909
Epoch 4/50
1650/1650 - 16s - 10ms/step - acc: 0.9491 - loss: 0.1403 - pr_auc: 0.9860 - precision: 0.9490 - recall: 0.9535 - roc_auc: 0.9870 - val_acc: 0.9593 - val_loss: 0.1169 - val_pr_auc: 0.9902 - val_precision: 0

/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1650/1650 - 17s - 11ms/step - acc: 0.9089 - loss: 0.2960 - pr_auc: 0.9585 - precision: 0.9140 - recall: 0.9106 - roc_auc: 0.9617 - val_acc: 0.9581 - val_loss: 0.1198 - val_pr_auc: 0.9889 - val_precision: 0.9551 - val_recall: 0.9652 - val_roc_auc: 0.9897
Epoch 2/50
1650/1650 - 16s - 10ms/step - acc: 0.9452 - loss: 0.1541 - pr_auc: 0.9835 - precision: 0.9450 - recall: 0.9500 - roc_auc: 0.9847 - val_acc: 0.9611 - val_loss: 0.1129 - val_pr_auc: 0.9901 - val_precision: 0.9598 - val_recall: 0.9659 - val_roc_auc: 0.9910
Epoch 3/50
1650/1650 - 16s - 10ms/step - acc: 0.9490 - loss: 0.1474 - pr_auc: 0.9838 - precision: 0.9483 - recall: 0.9541 - roc_auc: 0.9856 - val_acc: 0.9595 - val_loss: 0.1152 - val_pr_auc: 0.9900 - val_precision: 0.9584 - val_recall: 0.9645 - val_roc_auc: 0.9907
Epoch 4/50
1650/1650 - 16s - 10ms/step - acc: 0.9473 - loss: 0.1488 - pr_auc: 0.9839 - precision: 0.9462 - recall: 0.9529 - roc_auc: 0.9856 - val_acc: 0.9579 - val_loss: 0.1200 - val_pr_auc: 0.9903 - val_precision: 0

...

1650/1650 - 18s - 11ms/step - acc: 0.9114 - loss: 0.2735 - pr_auc: 0.9606 - precision: 0.9159 - recall: 0.9137 - roc_auc: 0.9636 - val_acc: 0.9596 - val_loss: 0.1173 - val_pr_auc: 0.9891 - val_precision: 0.9569 - val_recall: 0.9662 - val_roc_auc: 0.9902
Epoch 2/200
1650/1650 - 16s - 10ms/step - acc: 0.9447 - loss: 0.1570 - pr_auc: 0.9826 - precision: 0.9434 - recall: 0.9509 - roc_auc: 0.9842 - val_acc: 0.9601 - val_loss: 0.1145 - val_pr_auc: 0.9906 - val_precision: 0.9613 - val_recall: 0.9623 - val_roc_auc: 0.9908
Epoch 3/200
1650/1650 - 17s - 10ms/step - acc: 0.9476 - loss: 0.1477 - pr_auc: 0.9838 - precision: 0.9469 - recall: 0.9528 - roc_auc: 0.9856 - val_acc: 0.9564 - val_loss: 0.1243 - val_pr_auc: 0.9893 - val_precision: 0.9522 - val_recall: 0.9651 - val_roc_auc: 0.9898
Epoch 4/200
1650/1650 - 16s - 10ms/step - acc: 0.9474 - loss: 0.1487 - pr_auc: 0.9837 - precision: 0.9449 - recall: 0.9546 - roc_auc: 0.9855 - val_acc: 0.9598 - val_loss: 0.1156 - val_pr_auc: 0.9898 - val_precision

/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1650/1650 - 18s - 11ms/step - acc: 0.9243 - loss: 0.2239 - pr_auc: 0.9718 - precision: 0.9290 - recall: 0.9254 - roc_auc: 0.9737 - val_acc: 0.9595 - val_loss: 0.1175 - val_pr_auc: 0.9894 - val_precision: 0.9549 - val_recall: 0.9681 - val_roc_auc: 0.9904
Epoch 2/200
1650/1650 - 17s - 10ms/step - acc: 0.9487 - loss: 0.1441 - pr_auc: 0.9859 - precision: 0.9487 - recall: 0.9529 - roc_auc: 0.9870 - val_acc: 0.9598 - val_loss: 0.1154 - val_pr_auc: 0.9895 - val_precision: 0.9561 - val_recall: 0.9674 - val_roc_auc: 0.9907
Epoch 3/200
1650/1650 - 17s - 10ms/step - acc: 0.9503 - loss: 0.1391 - pr_auc: 0.9860 - precision: 0.9495 - recall: 0.9553 - roc_auc: 0.9872 - val_acc: 0.9582 - val_loss: 0.1160 - val_pr_auc: 0.9902 - val_precision: 0.9569 - val_recall: 0.9633 - val_roc_auc: 0.9907
Epoch 4/200
1650/1650 - 17s - 10ms/step - acc: 0.9504 - loss: 0.1406 - pr_auc: 0.9854 - precision: 0.9498 - recall: 0.9552 - roc_auc: 0.9869 - val_acc: 0.9561 - val_loss: 0.1222 - val_pr_auc: 0.9887 - val_precision

/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1650/1650 - 18s - 11ms/step - acc: 0.9115 - loss: 0.2886 - pr_auc: 0.9602 - precision: 0.9169 - recall: 0.9127 - roc_auc: 0.9637 - val_acc: 0.9611 - val_loss: 0.1173 - val_pr_auc: 0.9894 - val_precision: 0.9582 - val_recall: 0.9677 - val_roc_auc: 0.9904
Epoch 2/200
1650/1650 - 17s - 10ms/step - acc: 0.9450 - loss: 0.1560 - pr_auc: 0.9829 - precision: 0.9437 - recall: 0.9510 - roc_auc: 0.9844 - val_acc: 0.9589 - val_loss: 0.1177 - val_pr_auc: 0.9895 - val_precision: 0.9549 - val_recall: 0.9669 - val_roc_auc: 0.9904
Epoch 3/200
1650/1650 - 17s - 10ms/step - acc: 0.9477 - loss: 0.1473 - pr_auc: 0.9843 - precision: 0.9476 - recall: 0.9522 - roc_auc: 0.9858 - val_acc: 0.9601 - val_loss: 0.1154 - val_pr_auc: 0.9897 - val_precision: 0.9572 - val_recall: 0.9669 - val_roc_auc: 0.9906
Epoch 4/200


In [ ]:
# Filter only validation results
val_results = all_results_df[all_results_df.index.str.contains("val")]

# Sort by F1 Score in descending order
val_results_sorted = val_results.sort_values(by="F1 Score", ascending=False)

# Display the top results
print("🔝 Top Validation Results by F1 Score:")
display(val_results_sorted.head(10))

In [ ]:
model = tf.keras.models.load_model("best_mlp_model.h5", compile=False)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)  

In [ ]:
results = model.evaluate(X_test, y_test, verbose=0)
metrics = dict(zip(model.metrics_names, results))
print(metrics)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)     # default threshold

print(classification_report(y_test, y_pred, target_names=["Benign","Phish"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Benign","Phish"]).plot(cmap="Blues")
plt.show()

# ROC & PR curves
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC curve"); plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title("Precision–Recall curve"); plt.show()
